# Causal saliency — analysis & figures (v4.3 hybrid, **STUB**)

Reads the Parquet index + `.npz` arrays produced by
`causal_saliency_sweep.ipynb` and produces the statistical claims and
figures for the thesis "causal validation of saliency" section.

**Same caveat:** if the sweep was run on v4.3, the numbers here are
unreliable. This notebook is plumbing only; the figures it produces are
correct to the *method*, not to the underlying model.

## Sections
1. Load index + npz registry
2. ρ(saliency, occlusion) — per-superfamily distribution + bootstrap CIs
3. Deletion AUC: saliency vs random — paired Wilcoxon test
4. Sufficiency: how small a window keeps the prediction?
5. Position-relative saliency mass: 5'/3' ends vs middle
6. TIR overlap: does saliency concentrate in TIR ends for DNA superfamilies?
7. Cross-modal contribution: CNN tower vs GNN tower per class
8. Random-init negative control comparison (loaded from sibling notebook)
9. Summary table for thesis

In [ ]:
import os, sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def _find_repo(start: str) -> str:
    p = os.path.abspath(start)
    for _ in range(6):
        if os.path.isdir(os.path.join(p, "data", "vgp")):
            return p
        p = os.path.dirname(p)
    raise RuntimeError(f"could not locate repo root from {start}")
REPO = _find_repo(os.getcwd())

SWEEP_DIR = os.path.join(REPO, "model_result_interp", "interpretation_results",
                         "causal_saliency_hybrid", "sweep_v4.3")
ARR_DIR = os.path.join(SWEEP_DIR, "arrays")
INDEX_PATH = os.path.join(SWEEP_DIR, "index.parquet")
FIG_DIR = os.path.join(SWEEP_DIR, "figures")
os.makedirs(FIG_DIR, exist_ok=True)

assert os.path.exists(INDEX_PATH), f"Missing {INDEX_PATH} — run the sweep notebook first."
df = pd.read_parquet(INDEX_PATH)
print(f"index: {len(df)} rows, {df['superfamily'].nunique()} superfamilies, "
      f"{df['cls'].nunique()} classes")
df.head(3)

## 1. NPZ registry

Lazy loader so we don't put 10k arrays in memory at once.

In [ ]:
def _safe_name(header: str) -> str:
    return header.replace("/", "_").replace("#", "_").replace(":", "_")[:120]

def load_npz(header: str):
    p = os.path.join(ARR_DIR, _safe_name(header) + ".npz")
    return np.load(p)

# sanity
sample_header = df.iloc[0]["header"]
arrs = load_npz(sample_header)
print("array keys:", list(arrs.keys()))
print("sal shape:", arrs["sal"].shape, "| occN_drops shape:", arrs["occN_drops"].shape)

## 2. ρ(saliency, occlusion): per-superfamily distribution

In [ ]:
from scipy import stats

def bootstrap_median_ci(x, n_boot=2000, ci=95, seed=0):
    rng = np.random.default_rng(seed)
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    if x.size < 5:
        return float("nan"), float("nan"), float("nan")
    idx = rng.integers(0, x.size, size=(n_boot, x.size))
    meds = np.median(x[idx], axis=1)
    lo = float(np.percentile(meds, (100 - ci) / 2))
    hi = float(np.percentile(meds, 100 - (100 - ci) / 2))
    return float(np.median(x)), lo, hi

summary = []
for sf, sub in df.groupby("superfamily"):
    for col in ("rho_N", "rho_shuffle", "rho_reverse"):
        med, lo, hi = bootstrap_median_ci(sub[col].values)
        summary.append(dict(superfamily=sf, metric=col, median=med, ci_lo=lo, ci_hi=hi, n=len(sub)))
sf_summary = pd.DataFrame(summary)
sf_summary.head(10)

In [ ]:
# Boxplot per superfamily for ρ_N
order = sorted(df["superfamily"].unique())
data = [df.loc[df["superfamily"] == sf, "rho_N"].dropna().values for sf in order]
fig, ax = plt.subplots(figsize=(12, 5))
ax.boxplot(data, labels=order, showfliers=False)
ax.axhline(0, color="grey", lw=0.5)
ax.set_ylabel(r"$\rho_{\mathrm{spearman}}$(saliency, occlusion-N)")
ax.set_title("Saliency vs occlusion correlation per superfamily")
plt.xticks(rotation=70, ha="right", fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "rho_per_sf_boxplot.png"), dpi=150)
plt.show()

## 3. Deletion AUC: saliency vs random (paired Wilcoxon)

If saliency is causally meaningful, deleting saliency-ranked positions should
crash the logit faster than deleting random positions → **negative**
`auc_gap` (auc_random − auc_saliency).

In [ ]:
gaps = df["deletion_auc_gap"].dropna().values
W, p = stats.wilcoxon(gaps, alternative="less")  # H1: gap < 0
print(f"Wilcoxon signed-rank (H1: auc_gap < 0): W={W:.3g}, p={p:.3e}, n={gaps.size}")
print(f"median gap = {np.median(gaps):+.3f}, mean = {gaps.mean():+.3f}")

fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(gaps, bins=50, color="C0", alpha=0.7)
ax.axvline(0, color="k", lw=1)
ax.axvline(np.median(gaps), color="C3", lw=2, label=f"median {np.median(gaps):+.2f}")
ax.set_xlabel("deletion AUC gap (random − saliency)")
ax.set_ylabel("count")
ax.set_title("Saliency-ranked deletion is causal iff distribution shifts left of 0")
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "deletion_auc_gap.png"), dpi=150)
plt.show()

## 4. Sufficiency: how concentrated is the signal?

In [ ]:
# fraction of sequences where ANY 600bp window alone preserves predicted sf
fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(df["suff_max_logit"].dropna(), bins=50, color="C2", alpha=0.7)
ax.set_xlabel("max logit when keeping only one 600 bp window")
ax.set_ylabel("count")
ax.set_title("Sufficiency: a single 600 bp window's best logit per sequence")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "sufficiency_hist.png"), dpi=150)
plt.show()

## 5. Position-relative saliency mass

Average normalised saliency across sequences, by relative position
(0 = 5' end, 1 = 3' end). Quick view of whether the model attends to
ends or interior.

In [ ]:
N_BINS = 50
bin_acc = np.zeros((N_BINS,), dtype=np.float64)
bin_n = np.zeros((N_BINS,), dtype=np.int64)
for hd, slen in zip(df["header"].head(2000), df["seq_len"].head(2000)):  # cap for speed
    try:
        a = load_npz(hd)
    except FileNotFoundError:
        continue
    sal = np.abs(a["sal"])
    # sequence lives at positions [start, end) of canvas; we normalise within sequence
    # FIXED_LENGTH-padded; use seq_len to pick the central slice
    pad = max(0, sal.shape[0] - int(slen))
    start = pad // 2
    seq_sal = sal[start:start + int(slen)]
    if seq_sal.size < 10: continue
    seq_sal = seq_sal / (seq_sal.sum() + 1e-12)
    # bin to N_BINS
    idx = np.linspace(0, N_BINS, seq_sal.size, endpoint=False).astype(int)
    np.add.at(bin_acc, idx, seq_sal)
    np.add.at(bin_n, idx, 1)
mean_curve = bin_acc / np.maximum(bin_n, 1)

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(np.linspace(0, 1, N_BINS), mean_curve, lw=2)
ax.set_xlabel("relative position along sequence (5'→3')")
ax.set_ylabel("mean normalised |saliency|")
ax.set_title("Where in the sequence does saliency concentrate?")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "saliency_relpos.png"), dpi=150)
plt.show()

## 6. TIR overlap (DNA only)

For DNA-class sequences with a TIR annotation, test whether saliency
mass is enriched in the terminal 50–200 bp vs interior.

In [ ]:
TIR_LEN = 100
dna = df[(df["cls"] == "DNA") & (df["tir"] == 1)]
print(f"DNA-with-TIR sequences: {len(dna)}")

frac_in_terminal = []
for hd, slen in zip(dna["header"], dna["seq_len"]):
    try: a = load_npz(hd)
    except FileNotFoundError: continue
    sal = np.abs(a["sal"])
    pad = max(0, sal.shape[0] - int(slen))
    start = pad // 2
    seq_sal = sal[start:start + int(slen)]
    if seq_sal.size < 2 * TIR_LEN: continue
    total = seq_sal.sum() + 1e-12
    term = seq_sal[:TIR_LEN].sum() + seq_sal[-TIR_LEN:].sum()
    frac_in_terminal.append(term / total)

frac = np.array(frac_in_terminal)
expected = (2 * TIR_LEN) / dna["seq_len"].mean()  # rough null
print(f"mean fraction in {TIR_LEN}bp terminals: {frac.mean():.3f} (null ≈ {expected:.3f})")
W, p = stats.wilcoxon(frac - expected, alternative="greater")
print(f"Wilcoxon (H1: enrichment > null): W={W:.3g}, p={p:.3e}")

## 7. Cross-modal contribution

Bar chart of CNN-ablation drop vs GNN-ablation drop per class. Currently
`cross_modal_drop_*` columns are NaN until the sweep notebook implements
`_cross_modal_drop`.

In [ ]:
if df["cross_modal_drop_cnn"].notna().any():
    cm = df.groupby("cls")[["cross_modal_drop_cnn", "cross_modal_drop_gnn"]].median()
    cm.plot(kind="bar", figsize=(6, 4))
    plt.ylabel("median logit drop on tower ablation")
    plt.title("Cross-modal contribution per class")
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, "cross_modal_per_class.png"), dpi=150)
    plt.show()
else:
    print("cross_modal_drop columns are all NaN — implement _cross_modal_drop in the sweep notebook.")

## 8. Random-init negative control

Loads `sweep_v4.3_negctrl/index.parquet` (produced by a sibling sweep run
with an untrained model) and overlays its ρ distribution against the trained
model's. We expect the negative control to centre near 0.

In [ ]:
NEG_INDEX = os.path.join(REPO, "model_result_interp", "interpretation_results",
                         "causal_saliency_hybrid", "sweep_v4.3_negctrl",
                         "index.parquet")
if os.path.exists(NEG_INDEX):
    df_neg = pd.read_parquet(NEG_INDEX)
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.hist(df["rho_N"].dropna(), bins=40, alpha=0.5, label=f"trained (n={df['rho_N'].notna().sum()})")
    ax.hist(df_neg["rho_N"].dropna(), bins=40, alpha=0.5, label=f"random init (n={df_neg['rho_N'].notna().sum()})")
    ax.axvline(0, color="k", lw=0.5)
    ax.set_xlabel(r"$\rho_{\mathrm{spearman}}$(saliency, occlusion-N)")
    ax.set_ylabel("count")
    ax.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, "negctrl_overlay.png"), dpi=150)
    plt.show()
else:
    print(f"No negctrl index at {NEG_INDEX} — run causal_saliency_negctrl.ipynb (TBD).")

## 9. Thesis summary table

One-row-per-class summary suitable for the report. Headline numbers.

In [ ]:
agg = df.groupby("cls").agg(
    n=("header", "count"),
    rho_N_med=("rho_N", "median"),
    rho_shuffle_med=("rho_shuffle", "median"),
    deletion_gap_med=("deletion_auc_gap", "median"),
    suff_max_logit_med=("suff_max_logit", "median"),
)
agg

In [ ]:
# write to disk for inclusion in thesis
agg.to_csv(os.path.join(SWEEP_DIR, "thesis_summary_per_class.csv"))
sf_summary.to_csv(os.path.join(SWEEP_DIR, "rho_per_superfamily.csv"), index=False)
print("wrote summary CSVs to", SWEEP_DIR)